# Обучение ALS и сохранение эмбеддингов

1. Загружаем данные интеракций
2. Строим матрицу взаимодействий, делаем train/test split
3. Обучаем **две модели ALS** с разными гиперпараметрами
4. Считаем метрики (Precision@K, MAP@K) и логируем в MLflow
5. Записываем эмбеддинги лучшей модели в Redis

In [1]:
import pandas as pd
import numpy as np
from scipy.sparse import csr_matrix
import implicit
import mlflow
import redis
import json
import pickle

/home/dvpimanov/projects/MIPT_offline_recsys/venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## 1. Загрузка данных

In [2]:
interactions = pd.read_csv("../../../data/interactions.csv")
items = pd.read_csv("../../../data/items.csv")
users = pd.read_csv("../../../data/users.csv")

print(f"Interactions: {interactions.shape}")
print(f"Items: {items.shape}")
print(f"Users: {users.shape}")
interactions.head()

Interactions: (5476251, 5)
Items: (15963, 14)
Users: (840197, 5)


,user_id,item_id,last_watch_dt,total_dur,watched_pct
0,176549,9506,2021-05-11,4250,72.0
1,699317,1659,2021-05-29,8317,100.0
2,656683,7107,2021-05-09,10,0.0
3,864613,7638,2021-07-05,14483,100.0
4,964868,9506,2021-04-30,6725,100.0


## 2. Подготовка данных

In [3]:
# Фильтруем: берём только просмотры с watched_pct >= 10
interactions = interactions[interactions["watched_pct"] >= 10].copy()
print(f"После фильтрации: {interactions.shape}")

После фильтрации: (3721317, 5)


In [4]:
# Маппинги id <-> index
unique_users = interactions["user_id"].unique()
unique_items = interactions["item_id"].unique()

user_id_to_idx = {int(uid): idx for idx, uid in enumerate(unique_users)}
item_id_to_idx = {int(iid): idx for idx, iid in enumerate(unique_items)}

idx_to_user_id = {idx: uid for uid, idx in user_id_to_idx.items()}
idx_to_item_id = {idx: iid for iid, idx in item_id_to_idx.items()}

print(f"Users: {len(unique_users)}, Items: {len(unique_items)}")

Users: 762727, Items: 13851


In [5]:
# Train/test split по времени: последнее взаимодействие каждого пользователя -> test
interactions["last_watch_dt"] = pd.to_datetime(interactions["last_watch_dt"])
interactions = interactions.sort_values("last_watch_dt")

test_df = interactions.groupby("user_id").tail(1)
train_df = interactions.drop(test_df.index)

print(f"Train: {len(train_df)}, Test: {len(test_df)}")

Train: 2958590, Test: 762727


In [6]:
# Sparse-матрица для обучения (train)
train_matrix = csr_matrix(
    (
        (train_df["watched_pct"] / 100.0).values,
        (
            train_df["user_id"].map(user_id_to_idx).values,
            train_df["item_id"].map(item_id_to_idx).values,
        ),
    ),
    shape=(len(unique_users), len(unique_items)),
)

# Ground truth: что каждый пользователь посмотрел в тесте
test_ground_truth = {}
for uid, iid in zip(test_df["user_id"], test_df["item_id"]):
    u_idx = user_id_to_idx[int(uid)]
    i_idx = item_id_to_idx[int(iid)]
    test_ground_truth.setdefault(u_idx, set()).add(i_idx)

print(f"Train matrix: {train_matrix.shape}, nnz={train_matrix.nnz}")
print(f"Test users: {len(test_ground_truth)}")

Train matrix: (762727, 13851), nnz=2958590
Test users: 762727


## 3. Функции обучения и оценки

In [7]:
def get_factors(model):
    """Извлекает numpy-матрицы факторов (работает и для GPU-версии implicit)."""
    uf = model.user_factors
    itf = model.item_factors
    if hasattr(uf, "to_numpy"):
        uf = uf.to_numpy()
        itf = itf.to_numpy()
    return np.array(uf), np.array(itf)


def evaluate(model, train_mat, ground_truth, k=10):
    """Precision@K и MAP@K."""
    user_factors, item_factors = get_factors(model)
    precisions, aps = [], []

    for u_idx, true_items in ground_truth.items():
        scores = user_factors[u_idx] @ item_factors.T
        # Зануляем уже просмотренные в train
        scores[train_mat[u_idx].indices] = -np.inf
        top_k = np.argsort(scores)[::-1][:k]

        hits = len(set(top_k) & true_items)
        precisions.append(hits / k)

        cum_hits = 0
        ap = 0.0
        for rank, idx in enumerate(top_k, 1):
            if idx in true_items:
                cum_hits += 1
                ap += cum_hits / rank
        aps.append(ap / min(len(true_items), k))

    return {
        f"precision_at_{k}": float(np.mean(precisions)),
        f"map_at_{k}": float(np.mean(aps)),
    }

In [8]:
def train_and_log(run_name, factors, iterations, regularization):
    """Обучает ALS, считает метрики, логирует run в MLflow. Возвращает модель и метрики."""
    model = implicit.als.AlternatingLeastSquares(
        factors=factors,
        iterations=iterations,
        regularization=regularization,
        random_state=42,
    )
    model.fit(train_matrix)

    metrics = evaluate(model, train_matrix, test_ground_truth, k=10)

    with mlflow.start_run(run_name=run_name):
        mlflow.log_params({
            "factors": factors,
            "iterations": iterations,
            "regularization": regularization,
            "n_users": len(unique_users),
            "n_items": len(unique_items),
            "n_interactions": train_matrix.nnz,
        })
        mlflow.log_metrics(metrics)

        with open("/tmp/als_model.pkl", "wb") as f:
            pickle.dump(model, f)
        mlflow.log_artifact("/tmp/als_model.pkl")

        with open("/tmp/user_id_to_idx.json", "w") as f:
            json.dump({str(k): v for k, v in user_id_to_idx.items()}, f)
        with open("/tmp/item_id_to_idx.json", "w") as f:
            json.dump({str(k): v for k, v in item_id_to_idx.items()}, f)
        mlflow.log_artifact("/tmp/user_id_to_idx.json")
        mlflow.log_artifact("/tmp/item_id_to_idx.json")

        run_id = mlflow.active_run().info.run_id

    print(f"[{run_name}] run_id={run_id}")
    print(f"  params: factors={factors}, iterations={iterations}, reg={regularization}")
    print(f"  metrics: {metrics}")
    return model, metrics

## 4. Обучение двух моделей

| | factors | iterations | regularization | Идея |
|---|---|---|---|---|
| **simple** | 32 | 15 | 0.1 | Маленькая, сильная регуляризация |
| **expressive** | 128 | 30 | 0.01 | Большая, слабая регуляризация |

После обучения оба run-а появятся в MLflow UI — можно сравнить метрики и параметры.

In [9]:
mlflow.set_tracking_uri("http://localhost:5001")
mlflow.set_experiment("als_online")

model_simple, metrics_simple = train_and_log(
    run_name="als_simple",
    factors=32,
    iterations=15,
    regularization=0.1,
)

model_expressive, metrics_expressive = train_and_log(
    run_name="als_expressive",
    factors=128,
    iterations=30,
    regularization=0.01,
)

/home/dvpimanov/projects/MIPT_offline_recsys/venv/lib/python3.12/site-packages/implicit/cpu/als.py:95: RuntimeWarning: OpenBLAS is configured to use 12 threads. It is highly recommended to disable its internal threadpool by setting the environment variable 'OPENBLAS_NUM_THREADS=1' or by calling 'threadpoolctl.threadpool_limits(1, "blas")'. Having OpenBLAS use a threadpool can lead to severe performance issues here.
  check_blas_config()
100%|██████████| 15/15 [00:25<00:00,  1.68s/it]


🏃 View run als_simple at: http://localhost:5001/#/experiments/4/runs/c0e02ce9ac52464995f2a3be5dfcd54c
🧪 View experiment at: http://localhost:5001/#/experiments/4
[als_simple] run_id=c0e02ce9ac52464995f2a3be5dfcd54c
  params: factors=32, iterations=15, reg=0.1
  metrics: {'precision_at_10': 0.004754387874036189, 'map_at_10': 0.018699215873303063}


100%|██████████| 30/30 [01:18<00:00,  2.61s/it]


🏃 View run als_expressive at: http://localhost:5001/#/experiments/4/runs/05ded8a4b1da45b3ac6cf45cb99d0648
🧪 View experiment at: http://localhost:5001/#/experiments/4
[als_expressive] run_id=05ded8a4b1da45b3ac6cf45cb99d0648
  params: factors=128, iterations=30, reg=0.01
  metrics: {'precision_at_10': 0.004007462696351382, 'map_at_10': 0.01793561962433}


In [10]:
# Выбираем лучшую модель по MAP@10
if metrics_expressive["map_at_10"] >= metrics_simple["map_at_10"]:
    best_model = model_expressive
    best_name = "als_expressive"
else:
    best_model = model_simple
    best_name = "als_simple"

print(f"Best model: {best_name}")

user_embeddings, item_embeddings = get_factors(best_model)
print(f"User embeddings: {user_embeddings.shape}")
print(f"Item embeddings: {item_embeddings.shape}")

Best model: als_simple
User embeddings: (762727, 32)
Item embeddings: (13851, 32)


## 5. Запись эмбеддингов в Redis

Ключи:
- `als:user:{user_id}` — вектор пользователя
- `als:item:{item_id}` — вектор айтема
- `als:item_ids` — JSON-список всех item_id

In [11]:
r = redis.Redis(
    host="localhost",
    port=6379,
    password="recsys_redis_pass",
    decode_responses=True,
)
r.ping()

True

In [12]:
# Записываем user embeddings
pipe = r.pipeline()
for idx in range(len(unique_users)):
    uid = idx_to_user_id[idx]
    pipe.set(f"als:user:{uid}", json.dumps(user_embeddings[idx].tolist()))
pipe.execute()
print(f"Записано {len(unique_users)} user embeddings")

Записано 762727 user embeddings


In [13]:
# Записываем item embeddings
pipe = r.pipeline()
for idx in range(len(unique_items)):
    iid = idx_to_item_id[idx]
    pipe.set(f"als:item:{iid}", json.dumps(item_embeddings[idx].tolist()))
pipe.execute()
print(f"Записано {len(unique_items)} item embeddings")

Записано 13851 item embeddings


In [14]:
# Список всех item_id
all_item_ids = [idx_to_item_id[idx] for idx in range(len(unique_items))]
r.set("als:item_ids", json.dumps(all_item_ids))
print(f"Saved {len(all_item_ids)} item IDs")

Saved 13851 item IDs


In [15]:
# Проверка: dot product для первого пользователя и первого айтема
test_uid = idx_to_user_id[0]
test_iid = idx_to_item_id[0]

u_vec = np.array(json.loads(r.get(f"als:user:{test_uid}")))
i_vec = np.array(json.loads(r.get(f"als:item:{test_iid}")))

print(f"User {test_uid} dim={len(u_vec)}, Item {test_iid} dim={len(i_vec)}")
print(f"Dot product: {np.dot(u_vec, i_vec):.4f}")
print("\nГотово! Эмбеддинги в Redis, модели в MLflow.")

User 176549 dim=32, Item 9506 dim=32
Dot product: 0.2343

Готово! Эмбеддинги в Redis, модели в MLflow.
